# This is where we can train our own model

In [86]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch

Load the data for training

In [87]:
train_df = pd.read_csv("../data/train_images.csv")

In [88]:
class BirdDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

        # Convert labels to numeric if needed
        self.classes = sorted(self.df['label'].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open('../data' + row["image_path"]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = self.class_to_idx[row["label"]]
        return img, label


In [89]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = BirdDataset(train_df, transform=train_tfms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [90]:
num_classes = train_df['label'].unique()
len(num_classes)

200

Train the model

In [91]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 53 * 53, 400)
        self.fc2 = nn.Linear(400, 300)
        self.fc3 = nn.Linear(300, len(num_classes))

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

In [92]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.0003)

In [93]:
for epoch in range(15):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()

    print(f'[{epoch + 1}] loss: {running_loss/len(train_loader):.3f}')
    running_loss = 0.0

print('Finished Training')

[1] loss: 5.231
[2] loss: 4.985
[3] loss: 4.643
[4] loss: 4.227
[5] loss: 3.786
[6] loss: 3.258
[7] loss: 2.645
[8] loss: 1.992
[9] loss: 1.442
[10] loss: 0.975
[11] loss: 0.652
[12] loss: 0.418
[13] loss: 0.259
[14] loss: 0.188
[15] loss: 0.113
Finished Training


Run the model on test data

In [94]:
test_df = pd.read_csv("../data/test_images_path.csv")

In [95]:
test_dataset = BirdDataset(test_df, transform=train_tfms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [96]:
idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}

In [97]:
net.eval()

Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=44944, out_features=400, bias=True)
  (fc2): Linear(in_features=400, out_features=300, bias=True)
  (fc3): Linear(in_features=300, out_features=200, bias=True)
)

In [98]:
all_ids = test_df["id"].tolist()
all_preds = []

In [99]:
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())

In [100]:
predicted_labels = [idx_to_class[i] for i in all_preds]

In [101]:
output_df = pd.DataFrame({
    "id": all_ids,
    "label": predicted_labels
})

output_df.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv!")

Saved test_predictions.csv!


In [53]:
len(train_df)

3926